In [ ]:
import pandas as pd
import numpy as np
import requests
from urllib.parse import urlparse

# 1. Load the PhiUSIIL Dataset
data_path = "../data/PhiUSIIL_Phishing_URL_Dataset.csv"
df_full = pd.read_csv(data_path)
print(f"Total Rows Sampled: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
display(df.head(3))

Total Rows Sampled: 50000
Total Columns: 56


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,827598.txt,https://www.northcm.ac.th,24,www.northcm.ac.th,17,0,th,100.000000,0.800000,0.000510,...,0,0,1,47,3,4,69,7,115,1
1,oph00308.txt,https://unitedmartialartscenters.com/at0/mygov...,59,unitedmartialartscenters.com,28,0,com,33.522664,1.000000,0.522907,...,1,0,1,0,2,0,0,0,1,0
2,8112660.txt,https://email.mail1.onesignal.os.tc/c/ejwcz02u...,385,email.mail1.onesignal.os.tc,27,0,tc,3.199329,0.458333,0.000046,...,0,0,0,0,0,0,0,0,1,0


In [22]:
# Known URL shortening domains
SHORTENER_DOMAINS = {
    'bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly', 
    'is.gd', 'buff.ly', 'cutt.ly', 'rb.gy', 'shorturl.at'
}

def resolve_short_url(url, timeout=5):
    """
    Phase 1 Module: Unrolls shortened URLs via HTTP HEAD/GET redirect headers
    and extracts the true destination URL, root domain, and redirect hop count.
    """
    try:
        parsed_initial = urlparse(url)
        initial_domain = parsed_initial.netloc.lower().replace("www.", "")
        is_shortened = initial_domain in SHORTENER_DOMAINS
        
        # Trace HTTP redirects without downloading heavy page content
        response = requests.head(url, allow_redirects=True, timeout=timeout)
        final_url = response.url
        redirect_hops = len(response.history)
        final_domain = urlparse(final_url).netloc.lower().replace("www.", "")
        
        return {
            "Initial_URL": url,
            "Is_Shortener_Service": is_shortened,
            "Resolved_Final_URL": final_url,
            "Extracted_Root_Domain": final_domain,
            "NoOfURLRedirect": redirect_hops
        }
    except Exception as e:
        return {
            "Initial_URL": url,
            "Error": str(e),
            "Resolved_Final_URL": url,
            "NoOfURLRedirect": 0
        }

# Quick verification test (Safe offline/online demo for your guide)
print("Short-URL Redirect Resolution Module initialized successfully.")

Short-URL Redirect Resolution Module initialized successfully.


In [19]:
import pandas as pd
import tldextract

# 1. Invert label so Phishing = 1 (Positive Class) and Legitimate = 0
# In raw PhiUSIIL: 0 = Phishing, 1 = Legitimate
df_diag = df_full.copy()
df_diag['y_phish'] = (df_diag['label'] == 0).astype(int)

# 2. Extract both Hostname (raw 'Domain' column) and True Root Domain (eTLD+1)
df_diag['clean_host'] = (
    df_diag['Domain'].astype(str).str.lower().str.strip().str.replace(r'^www\.', '', regex=True)
)

# Extract eTLD+1 (e.g., 'sub.example.co.uk' -> 'example.co.uk')
def get_registered_domain(url_or_domain):
    ext = tldextract.extract(str(url_or_domain))
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}".lower()
    return str(url_or_domain).lower()

print("Extracting true root domains (eTLD+1)... (takes ~10-15 seconds on 235k rows)")
df_diag['root_domain'] = df_diag['clean_host'].apply(get_registered_domain)

# ==========================================
# REPORT 1: TRUE CLASS COUNTS (Phishing = 1)
# ==========================================
total_rows = len(df_diag)
phish_count = (df_diag['y_phish'] == 1).sum()
legit_count = (df_diag['y_phish'] == 0).sum()

print("\n=== 1. TRUE CLASS DISTRIBUTION (FULL DATASET) ===")
print(f"Total Rows          : {total_rows:,}")
print(f"Phishing (y = 1)    : {phish_count:,} ({phish_count/total_rows*100:.2f}%)")
print(f"Legitimate (y = 0)  : {legit_count:,} ({legit_count/total_rows*100:.2f}%)")

# ==========================================
# REPORT 2 & 3: OVERLAP ANALYSIS FUNCTION
# ==========================================
def analyze_domain_overlap(df, col_name, label_title):
    print(f"\n=== {label_title} (`{col_name}`) ===")
    n_total = len(df)
    n_unique = df[col_name].nunique()
    
    # Count how many times each domain appears
    counts = df[col_name].value_counts()
    repeated_domains = set(counts[counts > 1].index)
    
    # Flag rows that share a domain with at least one other row
    df['is_shared'] = df[col_name].isin(repeated_domains)
    shared_rows = df['is_shared'].sum()
    
    print(f"Unique Domains               : {n_unique:,} (out of {n_total:,} rows)")
    print(f"Domains appearing >1 time    : {len(repeated_domains):,}")
    print(f"Rows sharing a domain        : {shared_rows:,} / {n_total:,} ({shared_rows/n_total*100:.2f}%)")
    
    # Breakdown by Class
    print("\n--- Breakdown by Class ---")
    for cls_val, cls_name in [(1, "Phishing (y=1)"), (0, "Legitimate (y=0)")]:
        sub = df[df['y_phish'] == cls_val]
        n_cls = len(sub)
        n_cls_unique = sub[col_name].nunique()
        
        # Shared with ANY row in the dataset
        shared_any = sub['is_shared'].sum()
        
        # Shared within the SAME class
        cls_counts = sub[col_name].value_counts()
        cls_repeated = set(cls_counts[cls_counts > 1].index)
        shared_same_cls = sub[col_name].isin(cls_repeated).sum()
        
        print(f"{cls_name}:")
        print(f"  Total Rows                 : {n_cls:,}")
        print(f"  Unique Domains             : {n_cls_unique:,}")
        print(f"  Rows sharing domain (any)  : {shared_any:,} ({shared_any/n_cls*100:.2f}%)")
        print(f"  Rows sharing domain (class): {shared_same_cls:,} ({shared_same_cls/n_cls*100:.2f}%)")

    # Check cross-class domains (domains that have BOTH phishing and legitimate URLs)
    phish_domains = set(df[df['y_phish'] == 1][col_name])
    legit_domains = set(df[df['y_phish'] == 0][col_name])
    both_classes = phish_domains.intersection(legit_domains)
    rows_in_both = df[col_name].isin(both_classes).sum()
    print(f"\nDomains present in BOTH Phishing & Legit : {len(both_classes):,} domains ({rows_in_both:,} rows, {rows_in_both/n_total*100:.2f}%)")

# Run analysis on both raw 'Domain' (hostname) and true 'root_domain' (eTLD+1)
analyze_domain_overlap(df_diag, 'clean_host', "2A. RAW HOSTNAME OVERLAP")
analyze_domain_overlap(df_diag, 'root_domain', "2B. TRUE ROOT DOMAIN (eTLD+1) OVERLAP")

Extracting true root domains (eTLD+1)... (takes ~10-15 seconds on 235k rows)

=== 1. TRUE CLASS DISTRIBUTION (FULL DATASET) ===
Total Rows          : 235,795
Phishing (y = 1)    : 100,945 (42.81%)
Legitimate (y = 0)  : 134,850 (57.19%)

=== 2A. RAW HOSTNAME OVERLAP (`clean_host`) ===
Unique Domains               : 219,103 (out of 235,795 rows)
Domains appearing >1 time    : 5,871
Rows sharing a domain        : 22,563 / 235,795 (9.57%)

--- Breakdown by Class ---
Phishing (y=1):
  Total Rows                 : 100,945
  Unique Domains             : 84,311
  Rows sharing domain (any)  : 22,505 (22.29%)
  Rows sharing domain (class): 22,453 (22.24%)
Legitimate (y=0):
  Total Rows                 : 134,850
  Unique Domains             : 134,850
  Rows sharing domain (any)  : 58 (0.04%)
  Rows sharing domain (class): 0 (0.00%)

Domains present in BOTH Phishing & Legit : 58 domains (159 rows, 0.07%)

=== 2B. TRUE ROOT DOMAIN (eTLD+1) OVERLAP (`root_domain`) ===
Unique Domains               : 

In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

# 1. Print the missing Phishing Root-Domain (eTLD+1) overlap numbers (Point #2)
sub_phish = df_diag[df_diag['y_phish'] == 1]
p_root_counts = sub_phish['root_domain'].value_counts()
p_repeated_roots = set(p_root_counts[p_root_counts > 1].index)
p_shared_in_phish = sub_phish['root_domain'].isin(p_repeated_roots).sum()

all_root_counts = df_diag['root_domain'].value_counts()
all_repeated_roots = set(all_root_counts[all_root_counts > 1].index)
p_shared_any = sub_phish['root_domain'].isin(all_repeated_roots).sum()

print("=== 1. MISSING PHISHING ROOT-DOMAIN (eTLD+1) STATS ===")
print(f"Total Phishing Rows (y=1)                 : {len(sub_phish):,}")
print(f"Unique Phishing Root Domains (eTLD+1)     : {sub_phish['root_domain'].nunique():,}")
print(f"Phishing rows sharing root domain (any)   : {p_shared_any:,} ({p_shared_any/len(sub_phish)*100:.2f}%)")
print(f"Phishing rows sharing root domain (class) : {p_shared_in_phish:,} ({p_shared_in_phish/len(sub_phish)*100:.2f}%)")

print("\n--- Top 15 Most Repeated Phishing Root Domains (eTLD+1) ---")
print(p_root_counts.head(15).to_string())

# 2. Direct Train-to-Test Leakage & Fold Class-Balance Check (Points #3 & #5)
y_full = df_diag['y_phish'].values
groups_full = df_diag['root_domain'].values
X_dummy = np.zeros((len(df_diag), 1))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

=== 1. MISSING PHISHING ROOT-DOMAIN (eTLD+1) STATS ===
Total Phishing Rows (y=1)                 : 100,945
Unique Phishing Root Domains (eTLD+1)     : 43,512
Phishing rows sharing root domain (any)   : 62,828 (62.24%)
Phishing rows sharing root domain (class) : 62,741 (62.15%)

--- Top 15 Most Repeated Phishing Root Domains (eTLD+1) ---
root_domain
web.app             5754
firebaseapp.com     5594
repl.co             3754
weeblysite.com      3097
ipfs.io             1559
workers.dev         1438
square.site         1199
dweb.link            975
xsph.ru              928
pantheonsite.io      900
google.com           782
godaddysites.com     781
duckdns.org          642
webwave.dev          567
weebly.com           555


In [20]:

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

y_full = df_diag['y_phish'].values
groups_full = pd.factorize(df_diag['root_domain'])[0]   # integer IDs, done once
X_dummy = np.zeros((len(df_diag), 1))

def audit(splitter, name, use_groups):
    rows = []
    split_iter = splitter.split(X_dummy, y_full, groups_full) if use_groups else splitter.split(X_dummy, y_full)
    n_groups = groups_full.max() + 1
    for fold, (tr, te) in enumerate(split_iter, 1):
        seen = np.zeros(n_groups, dtype=bool)
        seen[groups_full[tr]] = True            # mark groups present in train
        leaked = seen[groups_full[te]]          # fast lookup
        te_y = y_full[te]
        rows.append({
            "Fold": fold, "Test Size": len(te),
            "Phish Rate (%)": round(te_y.mean()*100, 2),
            "Overall Leaked (%)": round(leaked.mean()*100, 2),
            "Phish Test Leaked (%)": round(leaked[te_y == 1].mean()*100, 2),
            "Legit Test Leaked (%)": round(leaked[te_y == 0].mean()*100, 2),
        })
    print(f"\n=== {name} ===")
    print(pd.DataFrame(rows).to_string(index=False))

audit(StratifiedKFold(5, shuffle=True, random_state=42), "RANDOM (StratifiedKFold)", False)
audit(StratifiedGroupKFold(5, shuffle=True, random_state=42), "GROUPED (StratifiedGroupKFold)", True)


=== RANDOM (StratifiedKFold) ===
 Fold  Test Size  Phish Rate (%)  Overall Leaked (%)  Phish Test Leaked (%)  Legit Test Leaked (%)
    1      47159           42.81               27.74                  61.47                   2.48
    2      47159           42.81               27.48                  60.77                   2.56
    3      47159           42.81               27.35                  60.71                   2.38
    4      47159           42.81               27.57                  60.92                   2.60
    5      47159           42.81               27.36                  60.84                   2.30

=== GROUPED (StratifiedGroupKFold) ===
 Fold  Test Size  Phish Rate (%)  Overall Leaked (%)  Phish Test Leaked (%)  Legit Test Leaked (%)
    1      47159           42.81                 0.0                    0.0                    0.0
    2      47159           42.81                 0.0                    0.0                    0.0
    3      47159           42.81   